# Delta Lake Assignment

## Incremental Data Processing Using Delta Lake

## Objective

The objective of this assignment is to perform incremental data processing using Delta Lake. The workflow includes loading a dataset into a Delta table, cleaning the data, creating an incremental dataset, applying a MERGE operation to update existing records and insert new records, validating the results, and displaying the final dataset.

## _Upload your dataset_
## 
In Databricks:

Catalog → Add data → Upload `File`

## Read the CSV and Display Results

Use the uploaded file path

In [0]:
df = spark.read.option("header", True)\
               .option("inferSchema", True)\
               .csv("/Volumes/workspace/default/dataset")

df.show(5)
df.printSchema()
df.count()

+---+------+---+--------+------+----------+------+
| ID|  Name|Age|    City|Gender|Department|Salary|
+---+------+---+--------+------+----------+------+
|  1| Priya| 20|Gurugram|Female|        HR| 44628|
|  2| Karan| 23|   Noida|  Male| Marketing| 57651|
|  3|  Amit| 20|  Mumbai|  Male|        HR| 63118|
|  4| Sonal| 20| Kolkata|  Male| Marketing| 57493|
|  5|Anjali| 34| Chennai|Female|        IT| 79729|
+---+------+---+--------+------+----------+------+
only showing top 5 rows
root
 |-- ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: integer (nullable = true)



57

## Cleaning Data

In [0]:
df = df.dropna()
df = df.dropDuplicates()
df.count()

53

## Save as a Delta table

In [0]:
table_name = "customer_master"

df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(table_name)



## Read delta table

In [0]:
customer_df = spark.read.table("customer_master")
customer_df.show(5)

+---+-----+---+----------+------+----------+------+
| ID| Name|Age|      City|Gender|Department|Salary|
+---+-----+---+----------+------+----------+------+
| 15| Riya| 27|     Noida|Female|        IT| 45010|
|  7|Arjun| 23|    Mumbai|Female|        IT| 53526|
| 31| Riya| 24|      Pune|Female| Marketing| 40821|
| 46|Nitin| 20|   Lucknow|Female|     Sales| 48694|
| 29|Priya| 31|Chandigarh|  Male|        IT| 45785|
+---+-----+---+----------+------+----------+------+
only showing top 5 rows


## Load incremental data

In [0]:
incremental_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/Volumes/workspace/default/dataset/incremental_data.csv")
)

incremental_df.show()

+---+------+---+---------+------+----------+------+
| ID|  Name|Age|     City|Gender|Department|Salary|
+---+------+---+---------+------+----------+------+
|  5|Anjali| 35|  Chennai|Female|        IT| 85000|
| 18|  Riya| 30|    Delhi|Female|        HR| 65000|
| 53|Aditya| 24|     Pune|  Male|   Finance| 50000|
| 54| Nisha| 28|Bengaluru|Female| Marketing| 62000|
| 55| Mohit| 31|   Jaipur|  Male|     Sales| 71000|
+---+------+---+---------+------+----------+------+



## Merge Operation

In [0]:
(
delta_table.alias("target")
.merge(
    incremental_df.alias("source"),
    "target.ID = source.ID"
)
.whenMatchedUpdate(
    set={
        "Name":"source.Name",
        "Age":"source.Age",
        "City":"source.City",
        "Gender":"source.Gender",
        "Department":"source.Department",
        "Salary":"source.Salary"
    }
)
.whenNotMatchedInsert(
    values={
        "ID":"source.ID",
        "Name":"source.Name",
        "Age":"source.Age",
        "City":"source.City",
        "Gender":"source.Gender",
        "Department":"source.Department",
        "Salary":"source.Salary"
    }
)
.execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Validate Results

After performing the merge operation, the final Delta table is validated by checking the total number of records and verifying that there are no duplicate IDs.

In [0]:
spark.read.table("customer_master").show()
spark.read.table("customer_master").count()

+---+------+---+----------+------+----------+------+
| ID|  Name|Age|      City|Gender|Department|Salary|
+---+------+---+----------+------+----------+------+
| 15|  Riya| 27|     Noida|Female|        IT| 45010|
|  7| Arjun| 23|    Mumbai|Female|        IT| 53526|
| 31|  Riya| 24|      Pune|Female| Marketing| 40821|
| 46| Nitin| 20|   Lucknow|Female|     Sales| 48694|
| 29| Priya| 31|Chandigarh|  Male|        IT| 45785|
| 33| Kavya| 34|   Kolkata|Female|        IT| 46246|
| 36|  Neha| 21|    Jaipur|  Male| Marketing| 45597|
| 38|  Isha| 27| Hyderabad|Female|        HR| 36181|
| 41| Priya| 27| Bengaluru|  Male| Marketing| 59400|
| 45|  Isha| 35| Bengaluru|Female|        IT| 40789|
| 28| Manoj| 20|   Chennai|Female|     Sales| 31276|
| 40|  Amit| 23|     Delhi|Female|   Finance| 82467|
| 44|  Riya| 20|    Mumbai|  Male|        HR| 56634|
|  6| Sneha| 33|    Jaipur|Female|        HR| 44110|
| 23|  Riya| 20|     Noida|  Male| Marketing| 79209|
| 17|Deepak| 30| Bengaluru|Female|     Sales| 

53

In [0]:
from pyspark.sql.functions import count

spark.read.table("customer_master") \
    .groupBy("ID") \
    .agg(count("*").alias("count")) \
    .filter("count > 1") \
    .show()

+---+-----+
| ID|count|
+---+-----+
|  5|    2|
| 18|    2|
+---+-----+



## Display Final Results

In [0]:
spark.read.table("customer_master").show()
spark.read.table("customer_master").count()

+---+------+---+----------+------+----------+------+
| ID|  Name|Age|      City|Gender|Department|Salary|
+---+------+---+----------+------+----------+------+
| 15|  Riya| 27|     Noida|Female|        IT| 45010|
|  7| Arjun| 23|    Mumbai|Female|        IT| 53526|
| 31|  Riya| 24|      Pune|Female| Marketing| 40821|
| 46| Nitin| 20|   Lucknow|Female|     Sales| 48694|
| 29| Priya| 31|Chandigarh|  Male|        IT| 45785|
| 33| Kavya| 34|   Kolkata|Female|        IT| 46246|
| 36|  Neha| 21|    Jaipur|  Male| Marketing| 45597|
| 38|  Isha| 27| Hyderabad|Female|        HR| 36181|
| 41| Priya| 27| Bengaluru|  Male| Marketing| 59400|
| 45|  Isha| 35| Bengaluru|Female|        IT| 40789|
| 28| Manoj| 20|   Chennai|Female|     Sales| 31276|
| 40|  Amit| 23|     Delhi|Female|   Finance| 82467|
| 44|  Riya| 20|    Mumbai|  Male|        HR| 56634|
|  6| Sneha| 33|    Jaipur|Female|        HR| 44110|
| 23|  Riya| 20|     Noida|  Male| Marketing| 79209|
| 17|Deepak| 30| Bengaluru|Female|     Sales| 

53

## Delta History


In [0]:
delta_table.history().show(truncate=False)

+-------+-------------------+--------------+-----------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Conclusion

The dataset was successfully loaded into a Delta table and cleaned by removing null values and duplicate records. An incremental dataset was created to simulate new and updated records. Using the Delta Lake MERGE operation, existing records were updated and new records were inserted efficiently. Finally, the results were validated by checking the row count, duplicate records, and Delta transaction history, demonstrating successful incremental data processing.